In [ ]:
# Ячейка 1: скомпилировать (один раз или при изменении .c)
!gcc -O3 -shared -fPIC dislocation_rings_v4.c -o dislocation_rings_v4.so

In [ ]:
from dislocation_rings_bind_v4 import make_ring_b, force_v4, make_dipole

In [ ]:
c=1.0;
a=1.0; # radius
print("ТЕСТ 1: осевая (climb) сила между петлёй N и анти-петлёй S,")
print("theta_b=0 у обеих (S=anti через charge_sign=-1), статичные петли,")
print("режим FERMI_NONE (без релятив. поправок -- чистая эластостатика).\n")
print(f"{'d/a':>6} {'Fz без glide':>16} {'Fz С glide':>16} {'подавление,%':>14}")

for dfrac in [0.1,0.3,0.5,1.0,1.5,2.0,3.0]:
    d = dfrac*a
    ring_N = make_ring_b((0, 0,  d/2), a, 0.0, 1.0, sign=1)
    ring_S = make_ring_b((0, 0, -d/2), a, 0.0, 1.0, sign=-1)
    F_noglide = force_v3([ring_N],[ring_S], 0.0, c, 3, 0)
    F_glide = force_v3([ring_N],[ring_S], 0.0, c, 3, 1)
    supp = 100*(1 - abs(F_glide[2])/(abs(F_noglide[2])+1e-30))
    print(f"{dfrac:6.2f} {F_noglide[2]:16.6f} {F_glide[2]:16.6f} {supp:14.2f}")

In [ ]:
print("Тест: диполь-диполь на больших R -- сравнение PLAIN / FILM_RETARDED / FILM_INSTANT")
print(f"{'R':>6} {'F*R^2 PLAIN':>14} {'F*R^2 FILM_RET':>16} {'F*R^2 FILM_INST':>17}")
for R in [5,10,20,40,80,160]:
    dipoleA = make_dipole((0,0,0))
    dipoleB = make_dipole((R,0,0))
    Fp = force_multi(dipoleA, dipoleB, 0.0, c, 0)
    # PLAIN
    Fr = force_multi(dipoleA, dipoleB, 0.0, c, 1)
    # FILM_RETARDED
    Fi = force_multi(dipoleA, dipoleB, 0.0, c, 2)
    # FILM_INSTANT
    magp = math.sqrt(sum(x*x for x in Fp))*R*R
    magr = math.sqrt(sum(x*x for x in Fr))*R*R
    magi = math.sqrt(sum(x*x for x in Fi))*R*R
    print(f"{R:6.1f} {magp:14.4f} {magr:16.4f} {magi:17.4f}")